# normal (masked-only) diffusion SFT — 1500 problems, 3 schedules, 2 epochs
### the controlled counterpart to the supervise-all curriculum run

**The one variable.** This notebook is a surgical diff of `superviseall_curriculum.ipynb`.
Everything error-prone is byte-identical: torchao removal, the transformers pin, the
`flash_attn` pure-PyTorch shim, `build_block_causal_mask`, `logits_of`, the metric helpers,
the per-sub-step backward, and the benchmark harness. The deliberate change is:

```
cfg.supervise_all = True   ->   False
```

which flips `sft_sequence_loss` from supervising all 32 block positions each sub-step to
supervising **only the still-masked ones** — the standard diffusion-LM objective. That branch
already exists in your engine (`sup = still.nonzero(...)`) and its loss normaliser
(`total_bpos`) already accounts for the shrinking supervised set, so the code path is the one
you wrote, not a new one.

**The second, disclosed change: data.** The curriculum run trained all three stages on the
*same* 500 problems (`stage_a + stage_b + stage_c` truncated to 500), i.e. 6 total passes over
one set. This run uses **three disjoint 500-problem pools** — 1500 distinct problems, 2 epochs
each. Forward-pass count per stage is unchanged, so **wall-clock is the same as your last run**;
only the data each stage sees is new. Consequence for the writeup: if this run behaves
differently, `supervise_all` and data-reuse are two candidate causes, and the per-tier +
repeat-4 signature is what separates them (see §14).

| stage | tokens/step | objective | data |
|---|---|---|---|
| **S1** | 4 | masked positions only | pool 1 — 500 problems × 2 epochs |
| **S2** | 2 | masked positions only | pool 2 — 500 problems × 2 epochs |
| **S3** | 1 | masked positions only | pool 3 — 500 problems × 2 epochs |

**Benchmark comparability.** Same `D["bench"]` 40 problems, same scorer, same schedules (2 and 1
tok/step), same `gen_budget`. `cfg.bench_include_raw` defaults to **False** — you already have
raw at 35% / 48%, and re-measuring it costs ~38 min for a number that cannot change. Stages are
evaluated **S3 first** (§13), because S3 is the single most informative row and you want it
before the deadline, not after S1 and S2 have burned two hours.

> **Isolation.** Writes to `sdar_normal_sft`; reads data from `sdar_posttrain`. The
> `sdar_superviseall` results and the OPSD notebook's files are never touched.

## 1 · GPU & Drive

In [ ]:
!nvidia-smi
import torch, platform
print("\nTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| Python:", platform.python_version())
assert torch.cuda.is_available(), "No GPU — Runtime ▸ Change runtime type."
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2 · Dependencies — torchao removed **before** transformers is imported
**If the transformers version changes, restart the runtime and re-run from the top.**

In [ ]:
import os, re, subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
for _n in [n for n in list(sys.modules) if n == "torchao" or n.startswith("torchao.")]:
    del sys.modules[_n]

if not os.path.isdir('/content/dLLM-RL'):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/Gen-Verse/dLLM-RL","/content/dLLM-RL"], check=True)

_FALLBACK = "transformers==4.51.3"; _spec = _FALLBACK
_req = "/content/dLLM-RL/requirements.txt"
if os.path.isfile(_req):
    m = re.search(r"^\s*transformers(\[[^\]]*\])?\s*([=<>!~].*?)\s*(?:#.*)?$", open(_req).read(), re.M)
    if m and m.group(2): _spec = "transformers" + (m.group(1) or "") + m.group(2).strip()
print("Pinning:", _spec)

!pip -q install "{_spec}" "accelerate>=0.33" "peft>=0.12" "datasets>=2.20" sentencepiece packaging

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import importlib.util
print("torchao present?", importlib.util.find_spec("torchao") is not None, "(must be False)")
import transformers; print("transformers:", transformers.__version__)
print("⚠️ If that version just CHANGED: Runtime ▸ Restart session, then run from the top.")


## 3 · Config

In [ ]:
from dataclasses import dataclass, field
from typing import Tuple
import os

@dataclass
class Cfg:
    # READ data from the original run; WRITE somewhere separate again.
    data_root: str  = "/content/drive/MyDrive/sdar_posttrain"
    drive_root: str = "/content/drive/MyDrive/sdar_normal_sft"

    model_id: str = "JetLM/SDAR-4B-Chat-b32"
    block_size: int = 32

    # ---- IDENTICAL to the supervise_all run (do not touch: this is the control) ----
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    lora_targets: Tuple[str, ...] = ("q_proj","k_proj","v_proj","o_proj",
                                     "gate_proj","up_proj","down_proj")
    schedules: Tuple[int, ...] = (4, 2, 1)     # tokens per step, coarse -> fine
    epochs: int = 2
    max_target_tokens: int = 384
    gen_budget: int = 1536
    lr: float = 1e-5
    grad_accum: int = 4
    warmup_steps: int = 10
    ckpt_every_samples: int = 20
    repeat4_max: float = 0.90
    bench_schedules: Tuple[int, ...] = (2, 1)

    # ---- THE EXPERIMENT ----------------------------------------------------
    supervise_all: bool = False                # <<< the single objective change

    # ---- data: three disjoint pools instead of one reused pool -------------
    n_per_stage: int = 500                     # 3 x 500 = 1500 distinct problems
    pool_seed: int = 0                         # deterministic stratified split
    require_complete: bool = False             # see §6: True = drop solutions that would be
                                               # truncated at max_target_tokens. Leaving this
                                               # False keeps supervise_all the ONLY difference.

    # ---- benchmark ---------------------------------------------------------
    bench_include_raw: bool = False            # raw is already measured: 35% @t2, 48% @t1
    bench_order: Tuple[str, ...] = ("s3_t1", "s1_t4", "s2_t2")   # most informative first

cfg = Cfg()
for sub in ("ckpt","results"):
    os.makedirs(os.path.join(cfg.drive_root, sub), exist_ok=True)
STATE_PATH = os.path.join(cfg.drive_root, "curriculum_state.json")
def ck(n): return os.path.join(cfg.drive_root, "ckpt", n)
def rs(n): return os.path.join(cfg.drive_root, "results", n)

N_TRAIN = cfg.n_per_stage
print(f"data from : {cfg.data_root}")
print(f"writing to: {cfg.drive_root}   (supervise_all + OPSD runs unaffected)")
print(f"objective : supervise_all={cfg.supervise_all}  <-- the experiment\n")
tot = 0
print(f"{'stage':<20}{'tok/step':>9}{'problems':>10}{'forwards':>12}{'hours':>8}")
for i, per in enumerate(cfg.schedules):
    f = N_TRAIN*cfg.epochs*(cfg.max_target_tokens//per); tot += f
    print(f"{'S'+str(i+1):<20}{per:>9}{N_TRAIN:>10}{f:>12,}{f*0.055/3600:>8.1f}")
print(f"{'TOTAL':<20}{'':>9}{N_TRAIN*3:>10}{tot:>12,}{tot*0.055/3600:>8.1f}")
print(f"\nSame forward count as the supervise_all run (500/stage either way) — only the")
print(f"problems differ. Benchmark adds ~2.5 h for three stages at 2 schedules")
print(f"(~3.2 h if you flip bench_include_raw back on).")
print(f"\nLEVERS if that does not fit before the deadline:")
print(f"  max_target_tokens 384 -> 256   ~{tot*0.055/3600*256/384:.1f} h  (halves S3 too)")
print(f"  epochs 2 -> 1                  ~{tot*0.055/3600/2:.1f} h  (but breaks the 2-epoch match)")
print(f"  n_per_stage 500 -> 300         ~{tot*0.055/3600*0.6:.1f} h")

## 4 · `flash_attn` → pure-PyTorch replacements

In [ ]:
import os, sys, types, importlib, importlib.abc, importlib.machinery, importlib.util
import torch, torch.nn.functional as F

USE_FLASH_ATTN = False
for _n in [n for n in list(sys.modules) if n=="flash_attn" or n.startswith("flash_attn.")]:
    del sys.modules[_n]
for _n in [n for n in list(sys.modules) if "modeling_sdar" in n]:
    del sys.modules[_n]

import transformers.dynamic_module_utils as _dmu
_real = getattr(_dmu, "_orig_get_imports", _dmu.get_imports)
_dmu._orig_get_imports = _real
def _pgi(fn):
    imp = list(_real(fn))
    if "flash_attn" in imp and os.path.basename(str(fn)).startswith("modeling_"):
        imp = [i for i in imp if i != "flash_attn"]
    return imp
_dmu.get_imports = _pgi

import transformers.utils as _tu
for _old, _c in {"LossKwargs": ["TransformersKwargs"]}.items():
    if not hasattr(_tu, _old):
        v = None
        for cand in _c:
            for mp in ("transformers.utils","transformers.processing_utils",
                       "transformers.modeling_utils","transformers"):
                try:
                    m = importlib.import_module(mp)
                    if hasattr(m, cand): v = getattr(m, cand); break
                except Exception: pass
            if v is not None: break
        setattr(_tu, _old, v if v is not None else type(_old, (dict,), {}))

def _rms_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None,
                 eps=1e-6, dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                 zero_centered_weight=False, return_dropout_mask=False, out_dtype=None,
                 out=None, residual_out=None):
    xdt = x.dtype
    if x1 is not None: x = x + x1
    base = ((x.float()+residual.float()) if residual_in_fp32 else (x+residual)) if residual is not None \
           else (x.float() if residual_in_fp32 else x)
    nr = base; xf = base.float()
    y = (xf*torch.rsqrt(xf.pow(2).mean(-1,keepdim=True)+eps)).to(xdt) * \
        ((1.0+weight) if zero_centered_weight else weight)
    if bias is not None: y = y + bias
    return (y, (nr if residual_in_fp32 else nr.to(xdt))) if prenorm else y

def _layer_norm_fn(x, weight, bias=None, residual=None, x1=None, weight1=None, bias1=None,
                   eps=1e-6, dropout_p=0.0, rowscale=None, prenorm=False, residual_in_fp32=False,
                   zero_centered_weight=False, is_rms_norm=False, return_dropout_mask=False,
                   out_dtype=None, out=None, residual_out=None):
    if is_rms_norm:
        return _rms_norm_fn(x, weight, bias, residual, x1, weight1, bias1, eps, dropout_p,
                            rowscale, prenorm, residual_in_fp32, zero_centered_weight,
                            return_dropout_mask, out_dtype, out, residual_out)
    xdt = x.dtype
    if x1 is not None: x = x + x1
    base = ((x.float()+residual.float()) if residual_in_fp32 else (x+residual)) if residual is not None \
           else (x.float() if residual_in_fp32 else x)
    nr = base; xf = base.float(); mu = xf.mean(-1,keepdim=True)
    y = ((xf-mu)*torch.rsqrt((xf-mu).pow(2).mean(-1,keepdim=True)+eps)).to(xdt) * \
        ((1.0+weight) if zero_centered_weight else weight)
    if bias is not None: y = y + bias
    return (y, (nr if residual_in_fp32 else nr.to(xdt))) if prenorm else y

def _expand_kv(k,v,nq):
    nk = k.shape[-2]
    if nk != nq:
        r = nq//nk; k = k.repeat_interleave(r,dim=-2); v = v.repeat_interleave(r,dim=-2)
    return k,v
def _flash_attn_func(q,k,v,dropout_p=0.0,softmax_scale=None,causal=False,window_size=(-1,-1),
                     softcap=0.0,alibi_slopes=None,deterministic=False,return_attn_probs=False,**kw):
    k,v=_expand_kv(k,v,q.shape[-2])
    o=F.scaled_dot_product_attention(q.transpose(1,2),k.transpose(1,2),v.transpose(1,2),
                                     is_causal=causal,scale=softmax_scale,dropout_p=0.0)
    return o.transpose(1,2)
def _flash_attn_qkvpacked_func(qkv,**kw):
    q,k,v=qkv.unbind(dim=2); return _flash_attn_func(q,k,v,**kw)
def _flash_attn_varlen_func(q,k,v,cu_seqlens_q,cu_seqlens_k,max_seqlen_q=None,max_seqlen_k=None,
                            dropout_p=0.0,softmax_scale=None,causal=False,**kw):
    cq,ck_=cu_seqlens_q.tolist(),cu_seqlens_k.tolist(); outs=[]
    for i in range(len(cq)-1):
        qi,ki,vi=q[cq[i]:cq[i+1]],k[ck_[i]:ck_[i+1]],v[ck_[i]:ck_[i+1]]
        ki,vi=_expand_kv(ki,vi,qi.shape[-2])
        oi=F.scaled_dot_product_attention(qi.transpose(0,1).unsqueeze(0),ki.transpose(0,1).unsqueeze(0),
                                          vi.transpose(0,1).unsqueeze(0),is_causal=causal,
                                          scale=softmax_scale,dropout_p=0.0)
        outs.append(oi.squeeze(0).transpose(0,1))
    return torch.cat(outs,0)
def _pad_input(hs,idx,b,s):
    out=hs.new_zeros(b*s,hs.shape[-1]); out[idx]=hs; return out.view(b,s,-1)
def _unpad_input(hs,am,*a,**k):
    sl=am.sum(-1).to(torch.int32); idx=torch.nonzero(am.flatten(),as_tuple=False).flatten()
    h=hs.reshape(-1,hs.shape[-1])[idx]; cu=torch.zeros(sl.numel()+1,dtype=torch.int32,device=hs.device)
    cu[1:]=torch.cumsum(sl,0); return h,idx,cu,int(sl.max().item())
def _index_first_axis(x,idx): return x.reshape(-1,*x.shape[1:])[idx]

class _RMSNormModule(torch.nn.Module):
    def __init__(self,hidden_size,eps=1e-6,**kw):
        super().__init__(); self.weight=torch.nn.Parameter(torch.ones(hidden_size)); self.eps=eps
    def forward(self,x,residual=None,prenorm=False,**kw):
        return _rms_norm_fn(x,self.weight,None,residual=residual,eps=self.eps,prenorm=prenorm)

_REG={"rms_norm_fn":_rms_norm_fn,"layer_norm_fn":_layer_norm_fn,"RMSNorm":_RMSNormModule,
      "LayerNorm":torch.nn.LayerNorm,"flash_attn_func":_flash_attn_func,
      "flash_attn_qkvpacked_func":_flash_attn_qkvpacked_func,
      "flash_attn_varlen_func":_flash_attn_varlen_func,"pad_input":_pad_input,
      "unpad_input":_unpad_input,"index_first_axis":_index_first_axis}
def _uns(n):
    def f(*a,**k): raise RuntimeError(f"flash_attn.{n} has no shim but was CALLED — report it.")
    return f
class _FM(types.ModuleType):
    def __getattr__(self,n):
        if n in _REG: return _REG[n]
        if n.startswith("__"): raise AttributeError(n)
        return _uns(n)
class _FF(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self,fn,path=None,target=None):
        if fn=="flash_attn" or fn.startswith("flash_attn."):
            return importlib.machinery.ModuleSpec(fn,self,is_package=True)
    def create_module(self,spec):
        m=_FM(spec.name); m.__spec__=spec; m.__path__=[]; m.__version__="0.0-shim"; return m
    def exec_module(self,m): pass

try:
    import flash_attn; USE_FLASH_ATTN=True; print("Real flash_attn present.")
except ImportError:
    if not any(isinstance(f,_FF) for f in sys.meta_path): sys.meta_path.insert(0,_FF())
    import flash_attn; print("✅ pure-PyTorch flash_attn replacements active.")
_t=torch.randn(2,4,8); _w=torch.randn(8)
assert torch.allclose(_rms_norm_fn(_t,_w,eps=1e-6),
                      _t*torch.rsqrt(_t.pow(2).mean(-1,keepdim=True)+1e-6)*_w, atol=1e-5)
print("✅ RMSNorm replacement matches reference math.")


## 5 · Metrics

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr

def repeat4(text):
    t = text.split()
    if len(t) < 4: return 0.0
    g = [tuple(t[i:i+4]) for i in range(len(t)-3)]
    return 1.0 - len(set(g))/len(g)

def extract_boxed(text):
    if not text: return None
    i = text.rfind("\\boxed")
    if i == -1: return None
    j = text.find("{", i)
    if j == -1: return None
    d = 0
    for k in range(j, len(text)):
        if text[k] == "{": d += 1
        elif text[k] == "}":
            d -= 1
            if d == 0: return text[j+1:k]
    return None

def has_complete_box(t): return extract_boxed(t) is not None

def _norm(s):
    if s is None: return None
    s = str(s).strip().replace(" ","")
    for a,b in (("\\left",""),("\\right",""),("\\dfrac","\\frac"),("\\tfrac","\\frac"),("$","")):
        s = s.replace(a,b)
    if s.startswith("\\text{") and s.endswith("}"): s = s[6:-1]
    return s

def answers_match(pred, gold):
    a,b = _norm(pred), _norm(gold)
    if a is None or b is None: return False
    if a == b: return True
    try: return abs(float(a)-float(b)) < 1e-6
    except Exception: return False

def structure_metrics(gen, ref, n_chunks=6):
    w = gen.split()
    if len(w) < n_chunks*4 or not ref.strip(): return float("nan"),float("nan"),float("nan")
    chunks = [" ".join(c) for c in np.array_split(np.array(w), n_chunks)]
    rw = ref.split(); tail = " ".join(rw[-max(20,len(rw)//4):])
    corpus = chunks + [tail, ref, gen]
    try: V = TfidfVectorizer(ngram_range=(1,2), min_df=1).fit_transform(corpus)
    except ValueError: return float("nan"),float("nan"),float("nan")
    C,tl,fr,fg = V[:n_chunks],V[n_chunks],V[n_chunks+1],V[n_chunks+2]
    sims = cosine_similarity(C,tl).ravel()
    prog = spearmanr(np.arange(n_chunks),sims).correlation if float(np.std(sims))>1e-9 else 0.0
    if prog is None or np.isnan(prog): prog = 0.0
    P = cosine_similarity(C); iu = np.triu_indices(n_chunks,k=1)
    return float(prog), float(P[iu].mean()), float(cosine_similarity(fg,fr)[0,0])

assert extract_boxed(r"a \boxed{1}, b \boxed{\frac{1}{2}}") == r"\frac{1}{2}"
assert answers_match(r"\dfrac{1}{2}", r"\frac{1}{2}") and not answers_match("3","4")
assert repeat4("a b c d "*6) > 0.5
print("✅ metric self-tests pass")


## 6 · Data — **three disjoint 500-problem pools** (1500 total)

The curriculum run concatenated `stage_a + stage_b + stage_c` and took the first 500, so all
three stages saw the same problems. Here the same union is split deterministically (seed 0,
**stratified by tier**, deduplicated by question) into three disjoint pools of 500 — so tier
composition is matched across stages and no problem is seen twice.

The benchmark 40 are excluded and the overlap assertion is kept.

**Truncation diagnostic (read the printout).** `max_target_tokens=384` truncates any longer
solution, and truncation removes the appended EOS — so every over-length sample teaches the
model a target that never terminates. This was true of the supervise_all run too, which is why
`require_complete` defaults to `False` (keeping the objective as the only difference). But if
the printed truncated fraction is large *and* this run also degrades with the non-termination
signature, truncation is a live confound and belongs in the paper's limitations. Setting
`cfg.require_complete = True` restricts each pool to solutions that fit and terminate — a
cleaner but no-longer-single-variable run.

In [ ]:
import json, os, random, re
from collections import Counter, defaultdict

DATA_PATH = os.path.join(cfg.data_root, "data_partitions.json")
assert os.path.isfile(DATA_PATH), f"{DATA_PATH} not found — check cfg.data_root."
D = json.load(open(DATA_PATH))

bench = D["bench"]
bq = {p["question"] for p in bench}

# ---- union of the original training partitions, deduped, benchmark removed ----
seen, union = set(), []
for key in ("stage_a", "stage_b", "stage_c"):
    for p in D.get(key, []):
        q = p["question"]
        if q in bq or q in seen: continue
        seen.add(q); union.append(p)
print(f"existing partitions (stage_a/b/c), deduped, benchmark removed: {len(union)}  "
      f"{dict(Counter(p['tier'] for p in union))}")
print(f"benchmark : {len(bench)}  {dict(Counter(p['tier'] for p in bench))}")

# ---- strip raw R1 <think>...</think> reasoning traces, if present -------------
# DeepMath-103K's r1_solution_i columns are raw DeepSeek-R1 rollouts: a (often huge)
# reasoning trace wrapped in <think>...</think>, followed by the actual final solution.
# "shortest of 3" alone barely helps if all three ran long reasoning independently of
# solution length — the bloat is the thinking, not the answer. Stripping it is also the
# more correct SFT target: the model's own prompt asks for a step-by-step *solution*
# ending in \boxed{}, not a reproduction of R1's internal scratchpad, and a diffusion LM
# has no mechanism analogous to sequential "wait, let me reconsider" self-correction
# anyway. Applied here to BOTH the existing 500 and freshly sourced problems, so the
# fix is uniform regardless of which pool a sample came from.
_THINK_RE = re.compile(r"<think>.*?</think>", re.S)
_had_think = 0
for p in union:
    if "<think>" in p["solution"]:
        stripped = _THINK_RE.sub("", p["solution"]).strip()
        if "\\boxed" in stripped:            # only replace if the boxed answer survives the strip
            p["solution"] = stripped; _had_think += 1
if _had_think:
    print(f"stripped <think>...</think> from {_had_think}/{len(union)} existing solutions")

need = cfg.n_per_stage * 3

# ---- tokenizer, loaded once, reused for require_complete filtering + the diagnostic below ----
from transformers import AutoTokenizer as _AT
_tk = _AT.from_pretrained(cfg.model_id, trust_remote_code=True)
def _fits(sol):
    return (not cfg.require_complete) or \
           len(_tk(sol, add_special_tokens=False).input_ids) < cfg.max_target_tokens

if cfg.require_complete:
    before = len(union)
    union = [p for p in union if _fits(p["solution"])]
    print(f"require_complete=True: kept {len(union)}/{before} of the existing pool")

# ---- data_partitions.json only holds ~500 problems total -> source the shortfall -------------
# from the raw DeepMath-103K dataset directly (same source data_partitions.json was built from).
DATASET_ID   = "zwhe99/DeepMath-103K"
EASY_MAX     = 4.0    # difficulty <= EASY_MAX            -> easy
MED_MAX      = 6.0    # EASY_MAX  < difficulty <= MED_MAX -> medium   (else hard)
INCLUDE_EASY = False  # existing 500 are 0% easy (320 medium / 180 hard) — reads as a deliberate
                      # choice to train on medium/hard only and reserve easy for the benchmark.
                      # Set True to also draw easy problems into the training pools.

if len(union) < need:
    print(f"\n⚠️ only {len(union)} usable problems on disk — {need - len(union)} short of "
          f"the {need} needed for 3 FRESH {cfg.n_per_stage}-problem pools.")
    print(f"Sourcing the shortfall from {DATASET_ID}...\n")
    from datasets import load_dataset
    raw = load_dataset(DATASET_ID, split="train")
    Q_COL = "question" if "question" in raw.column_names else "problem"
    SOL_CANDS = [c for c in ("r1_solution_1","r1_solution_2","r1_solution_3","solution")
                 if c in raw.column_names]
    assert "difficulty" in raw.column_names, \
        f"no difficulty column in {DATASET_ID} — inspect raw.column_names and fix tiering above"

    hist = Counter(round(float(x)) for x in raw["difficulty"])
    print("raw difficulty histogram (rounded):", dict(sorted(hist.items())))
    print(f"tiering: easy <= {EASY_MAX}, medium <= {MED_MAX}, else hard  "
          f"(adjust EASY_MAX/MED_MAX above if this looks wrong, then re-run this cell)\n")

    def tier_of(d):
        d = float(d)
        return "easy" if d <= EASY_MAX else ("medium" if d <= MED_MAX else "hard")

    def shortest_solution(ex):
        # Strip <think>...</think> from EACH candidate before ranking by length — pick
        # the shortest FINAL solution among the three, not the shortest raw rollout
        # (a rollout can be short-thinking-but-wrong or long-thinking-but-equivalent;
        # what should decide "shortest" is the actual solution text, not the scratchpad).
        cands = []
        for c in SOL_CANDS:
            s = ex.get(c)
            if not isinstance(s, str): continue
            s = _THINK_RE.sub("", s).strip()
            if "\\boxed" in s: cands.append(s)
        return min(cands, key=len) if cands else None

    order = list(range(len(raw))); random.Random(cfg.pool_seed + 99).shuffle(order)
    added, think_seen = 0, 0
    for i in order:
        if len(union) >= need: break
        ex = raw[i]; q = ex[Q_COL]
        if q in bq or q in seen: continue
        t = tier_of(ex["difficulty"])
        if t == "easy" and not INCLUDE_EASY: continue
        if any(isinstance(ex.get(c), str) and "<think>" in ex[c] for c in SOL_CANDS):
            think_seen += 1
        sol = shortest_solution(ex)
        if sol is None or not _fits(sol): continue
        seen.add(q); union.append({"tier": t, "question": q, "solution": sol})
        added += 1
    print(f"sourced {added} fresh problems from {DATASET_ID} "
          f"({think_seen} candidates seen had a <think> block, stripped before length-ranking)")
    del raw

print(f"final union: {len(union)}  {dict(Counter(p['tier'] for p in union))}")
del _tk
assert len(union) >= need, (f"only {len(union)} problems available even after sourcing more, "
                            f"need {need}. Lower cfg.n_per_stage, set INCLUDE_EASY=True above, "
                            f"or check DATASET_ID / EASY_MAX / MED_MAX.")

# ---- deterministic stratified split into three disjoint pools -----------------
by_tier = defaultdict(list)
for p in union: by_tier[p["tier"]].append(p)
rng = random.Random(cfg.pool_seed)
for t in by_tier: rng.shuffle(by_tier[t])

pools = [[], [], []]
for t, items in by_tier.items():                 # deal round-robin -> matched tier mix
    for j, p in enumerate(items):
        if min(len(x) for x in pools) >= cfg.n_per_stage: break
        k = j % 3
        if len(pools[k]) < cfg.n_per_stage: pools[k].append(p)
        elif len(pools[(k+1) % 3]) < cfg.n_per_stage: pools[(k+1) % 3].append(p)
        elif len(pools[(k+2) % 3]) < cfg.n_per_stage: pools[(k+2) % 3].append(p)
for k in range(3):
    rng.shuffle(pools[k])
    assert len(pools[k]) == cfg.n_per_stage, f"pool {k} has {len(pools[k])}"

qs = [ {p["question"] for p in pl} for pl in pools ]
assert not (qs[0] & qs[1]) and not (qs[0] & qs[2]) and not (qs[1] & qs[2]), "pools overlap!"
assert not any(q & bq for q in qs), "benchmark leaked into training data!"
for k, pl in enumerate(pools):
    print(f"  pool {k+1}: {len(pl)}  {dict(Counter(p['tier'] for p in pl))}")
print("disjoint ✓   benchmark-clean ✓")

json.dump([[p["question"] for p in pl] for pl in pools],
          open(rs("stage_pools.json"), "w"), indent=1)

# ---- truncation diagnostic: sweep candidate budgets instead of guessing one ---
from transformers import AutoTokenizer as _AT2
_tk = _AT2.from_pretrained(cfg.model_id, trust_remote_code=True)
_sample = [p for pl in pools for p in pl][:300]
_lens = [len(_tk(p["solution"], add_special_tokens=False).input_ids) for p in _sample]
print(f"\nsolution length over {len(_sample)} sampled problems: "
      f"median {sorted(_lens)[len(_lens)//2]}, p90 {sorted(_lens)[int(.9*len(_lens))]}")

print(f"\n{'max_target_tokens':>18}{'% truncated':>13}{'% surviving w/ require_complete':>34}")
for cand in (384, 768, 1152, 1536, 2048, 3072):
    trunc = sum(1 for L in _lens if L >= cand) / len(_lens)
    survive = 1 - trunc
    print(f"{cand:>18}{trunc:>12.0%}{survive:>33.0%}   "
          f"(~{(cand/384)*10.3:.1f} h projected train time)")

_trunc384 = sum(1 for L in _lens if L >= cfg.max_target_tokens) / len(_lens)
print(f"\nat cfg.max_target_tokens={cfg.max_target_tokens}: {_trunc384:.0%} truncated")
if _trunc384 > 0.5:
    print("  ⚠️ most targets are truncated and therefore have NO EOS — every such sample")
    print("     teaches 'do not terminate'. Identical in the supervise_all run, so the")
    print("     comparison stays valid, but record this in the paper's limitations.")
    print("  Raising max_target_tokens barely fixes this (see table above: even 1536 still")
    print("  truncates most solutions) while multiplying training time roughly linearly.")
    print("  cfg.require_complete=True fixes the actual mechanism instead — it SELECTS")
    print("  solutions that already fit and terminate rather than cutting long ones, at")
    print("  no extra compute cost. Re-run §6 with require_complete=True and check the")
    print("  'kept N/M' line above lands near cfg.n_per_stage*3 before deciding it's viable.")
del _tk, _sample, _lens

## 7 · Model, LoRA, mask
`modeling_sdar.py`'s outer `forward()` has `_update_causal_mask` commented out, so the block-causal
mask is entirely our responsibility. `fuse_cross_entropy` is forced off — it returns `logits=None`
whenever `self.training` is True.

In [ ]:
import torch, contextlib, gc, math, time, transformers
from packaging import version as _v
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel

_dt_kw = "dtype" if _v.parse(transformers.__version__.split("+")[0]) >= _v.parse("4.56.0") \
         else "torch_dtype"

def build_block_causal_mask(seq_len, prompt_len, block_size, device):
    idx = torch.arange(seq_len, device=device)
    is_resp = idx >= prompt_len
    blk = torch.where(is_resp, (idx-prompt_len)//block_size, idx)
    qr,kr = is_resp.unsqueeze(1), is_resp.unsqueeze(0)
    qi,ki = idx.unsqueeze(1), idx.unsqueeze(0)
    qb,kb = blk.unsqueeze(1), blk.unsqueeze(0)
    return ((~qr)&(ki<=qi)) | (qr&(~kr)) | (qr&kr&(kb<=qb))

def load_base():
    tok = AutoTokenizer.from_pretrained(cfg.model_id, trust_remote_code=True)
    m = AutoModelForCausalLM.from_pretrained(
        cfg.model_id, trust_remote_code=True, device_map="cuda",
        attn_implementation="flash_attention_2" if USE_FLASH_ATTN else "sdpa",
        **{_dt_kw: torch.bfloat16})
    if hasattr(m.config,"fuse_cross_entropy"): m.config.fuse_cross_entropy = False
    m.gradient_checkpointing_disable()
    return m, tok

def fresh_lora(m):
    return get_peft_model(m, LoraConfig(
        r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.lora_targets), bias="none", task_type="CAUSAL_LM"))

def build_invalid(m, tok):
    vs, rv = m.config.vocab_size, len(tok)
    inv = torch.zeros(vs, dtype=torch.bool)
    if rv < vs: inv[rv:] = True
    mid = getattr(tok,"mask_token_id",None) or 151669
    inv[mid] = True
    return inv.to(m.device), mid

def student_prompt(tok, q):
    return tok.apply_chat_template(
        [{"role":"user","content": f"{q}\nPlease reason step by step, and put your final answer "
                                   f"within \\boxed{{}}."}], tokenize=False, add_generation_prompt=True)

def teacher_prompt(tok, q, sol):
    c = (f"{q}\n\nHere is a reference solution:\n{sol}\n\nAfter understanding the reference "
         f"solution, solve the problem yourself.\nPlease reason step by step, and put your final "
         f"answer within \\boxed{{}}.")
    return tok.apply_chat_template([{"role":"user","content":c}], tokenize=False,
                                   add_generation_prompt=True)

def logits_of(model, ids, prompt_len, teacher=False):
    mask = build_block_causal_mask(ids.shape[1], prompt_len, cfg.block_size, ids.device)[None,None]
    ctx = model.disable_adapter() if teacher else contextlib.nullcontext()
    with ctx:
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            return model(input_ids=ids, attention_mask=mask).logits

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()
    print(f"    VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

def save_adapters(model, name):
    d = ck(name); os.makedirs(d, exist_ok=True); model.save_pretrained(d)

def adapters_exist(name):
    return os.path.isfile(os.path.join(ck(name),"adapter_model.safetensors")) or \
           os.path.isfile(os.path.join(ck(name),"adapter_model.bin"))

DEFAULT_STATE = {s:{"done":False,"epoch":0,"sample":0,"opt_step":0}
                 for s in ("stage_a","stage_b","stage_c","opsd1","opsd2","bench")}
def load_state():
    if os.path.isfile(STATE_PATH):
        st = json.load(open(STATE_PATH))
        for k,v in DEFAULT_STATE.items(): st.setdefault(k, dict(v))
        return st
    return {k:dict(v) for k,v in DEFAULT_STATE.items()}
def save_state(st): json.dump(st, open(STATE_PATH,"w"), indent=1)

STATE = load_state()
print("model helpers ready.")


## 8 · State

In [ ]:
import json, os, torch

STAGE_NAMES = [f"s{i+1}_t{per}" for i, per in enumerate(cfg.schedules)]
DEFAULT_STATE = {s: {"done": False, "epoch": 0, "sample": 0, "opt_step": 0} for s in STAGE_NAMES}

def load_state():
    if os.path.isfile(STATE_PATH):
        st = json.load(open(STATE_PATH))
        for k, v in DEFAULT_STATE.items(): st.setdefault(k, dict(v))
        return st
    return {k: dict(v) for k, v in DEFAULT_STATE.items()}
def save_state(st): json.dump(st, open(STATE_PATH, "w"), indent=1)
def save_adapters(model, name):
    d = ck(name); os.makedirs(d, exist_ok=True); model.save_pretrained(d)
def adapters_exist(name):
    return os.path.isfile(os.path.join(ck(name), "adapter_model.safetensors")) or \
           os.path.isfile(os.path.join(ck(name), "adapter_model.bin"))

STATE = load_state()
print("STAGES:", STAGE_NAMES)
for k, v in STATE.items():
    print(f"  {k:<10} done={str(v['done']):<5} epoch={v['epoch']} sample={v['sample']}")
print("\ncheckpoints:", sorted(os.listdir(ck(""))) if os.path.isdir(ck("")) else [])


## 9 · SFT engine — **masked-only**, backward per sub-step

`sft_sequence_loss` is unchanged from the supervise_all notebook. With
`cfg.supervise_all=False` it takes the other branch: at each sub-step the supervised set is
`still.nonzero()` — the positions that are still `[MASK]` in the model's input — and the
normaliser `total_bpos = n_sub*bs - per*n_sub*(n_sub-1)/2` is the exact count of those
supervised positions summed over sub-steps, so per-position loss scale (and therefore the
effective learning rate at `lr=1e-5`) matches.

Backward still runs once per sub-step: mathematically identical to once per block, but peak
memory stays at one forward graph instead of `32/tokens_per_step` of them — the fix for the
earlier 1-tok/step OOM.

`run_stage` now takes the stage's own `pool` as an argument (the three stages no longer share
one `train_pool`). Everything else — resume logic, checkpoint cadence, cosine schedule,
per-sample JSONL logging — is unchanged.

In [ ]:
import torch, torch.nn.functional as F, math, time, json
from transformers import get_cosine_schedule_with_warmup

def sft_sequence_loss(model, tok, invalid, mask_id, question, solution,
                      tokens_per_step, loss_scale, supervise_all):
    dev = model.device
    p_ids = tok(student_prompt(tok, question), return_tensors="pt").input_ids.to(dev)
    s_ids = tok(solution, return_tensors="pt", add_special_tokens=False).input_ids.to(dev)
    if tok.eos_token_id is not None:
        s_ids = torch.cat([s_ids, torch.tensor([[tok.eos_token_id]], device=dev,
                                               dtype=s_ids.dtype)], dim=1)
    s_ids = s_ids[:, :cfg.max_target_tokens]
    B, per = cfg.block_size, tokens_per_step
    prompt_len = p_ids.shape[1]
    ctx = p_ids
    running, npos, nblk = 0.0, 0, 0

    for b0 in range(0, s_ids.shape[1], B):
        gold = s_ids[:, b0:b0+B]; bs = gold.shape[1]
        if bs == 0: break
        n_sub = math.ceil(bs/per)
        total_bpos = bs*n_sub if supervise_all else (n_sub*bs - per*n_sub*(n_sub-1)//2)
        revealed = torch.full((1,bs), mask_id, device=dev, dtype=ctx.dtype)
        still = torch.ones(bs, dtype=torch.bool, device=dev)
        nblk += 1
        while bool(still.any()):
            work = torch.cat([ctx, revealed], dim=1)
            pos = torch.arange(ctx.shape[1], ctx.shape[1]+bs, device=dev)
            lg = logits_of(model, work, prompt_len)[0, pos, :].float().masked_fill(invalid, float("-inf"))
            sup = torch.arange(bs, device=dev) if supervise_all else still.nonzero(as_tuple=True)[0]
            step_loss = F.cross_entropy(lg[sup], gold[0, sup], reduction="sum") / total_bpos
            (step_loss * loss_scale).backward()
            running += step_loss.item(); npos += int(sup.numel())
            conf = torch.log_softmax(lg,-1).max(-1).values.masked_fill(~still, float("-inf"))
            k = min(per, int(still.sum().item()))
            sel = conf.topk(k).indices
            revealed[0, sel] = gold[0, sel]      # teacher-force GOLD
            still[sel] = False
        ctx = torch.cat([ctx, gold.detach()], dim=1)
    return running/max(1,nblk), npos, nblk

def run_stage(stage, source, tokens_per_step, pool):
    # Independent per-stage runner: own model load, own optimizer, own cleanup.
    # `pool` is this stage's own 500 problems (stages no longer share one pool).
    st = STATE[stage]
    if st["done"]:
        print(f"{stage}: already done — skipping."); return
    resume = f"{stage}_inprogress" if adapters_exist(f"{stage}_inprogress") else source
    base, tok = load_base()
    if resume is None:
        print(f"{stage}: fresh LoRA"); model = fresh_lora(base)
    else:
        assert adapters_exist(resume), f"checkpoint '{resume}' missing"
        print(f"{stage}: loading adapters from {resume}")
        model = PeftModel.from_pretrained(base, ck(resume), is_trainable=True)
    model.train(); model.print_trainable_parameters()
    invalid, mask_id = build_invalid(model, tok)
    trainable = [q for q in model.parameters() if q.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=cfg.lr)
    sched = get_cosine_schedule_with_warmup(
        opt, cfg.warmup_steps, max(1, math.ceil(len(pool)*cfg.epochs/cfg.grad_accum)))
    for _ in range(st["opt_step"]): sched.step()
    opt.zero_grad(set_to_none=True)

    print(f"\n=== {stage} | {len(pool)} problems x {cfg.epochs} epochs | "
          f"{tokens_per_step} tok/step | supervise_all={cfg.supervise_all} ===")
    if st["epoch"] or st["sample"]:
        print(f"  resuming at epoch {st['epoch']+1}, sample {st['sample']}")
    acc, t0 = 0, time.time()
    for ep in range(st["epoch"], cfg.epochs):
        start = st["sample"] if ep == st["epoch"] else 0
        for i in range(start, len(pool)):
            p = pool[i]
            loss, npos, nblk = sft_sequence_loss(model, tok, invalid, mask_id,
                                                 p["question"], p["solution"], tokens_per_step,
                                                 1.0/cfg.grad_accum, cfg.supervise_all)
            acc += 1
            if acc == cfg.grad_accum:
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
                acc = 0; st["opt_step"] += 1
            st["epoch"], st["sample"] = ep, i+1
            save_state(STATE)
            with open(rs(f"{stage}_log.jsonl"), "a") as f:
                f.write(json.dumps({"ep":ep,"i":i,"loss":loss,"npos":npos,"nblk":nblk})+"\n")
            if (i+1) % 20 == 0:
                el = (time.time()-t0)/60
                print(f"  e{ep+1} [{i+1:>4}/{len(pool)}] loss {loss:.4f} | {nblk:>2} blk "
                      f"{npos:>5} sup | {el:.1f} min | "
                      f"{torch.cuda.memory_allocated()/1e9:.1f} GB")
            if (i+1) % cfg.ckpt_every_samples == 0:
                save_adapters(model, f"{stage}_inprogress")
        st["sample"] = 0; st["epoch"] = ep+1
        save_state(STATE); save_adapters(model, f"{stage}_inprogress")
        print(f"  -- epoch {ep+1}/{cfg.epochs} done --")
    if acc > 0:
        torch.nn.utils.clip_grad_norm_(trainable, 1.0); opt.step(); sched.step()
    st["done"] = True; save_state(STATE); save_adapters(model, stage)
    print(f"✅ {stage} complete -> {ck(stage)}")
    del model, base, tok, invalid, opt, sched, trainable
    import gc; gc.collect(); torch.cuda.empty_cache()
    print(f"    VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

assert cfg.supervise_all is False, "this notebook is the masked-only arm — check §3"
print("SFT engine ready (MASKED-ONLY objective, per-sub-step backward).")


## 10 · Stage S1 — 4 tokens/step, pool 1
Fresh LoRA on the raw base. Independent cell: own model load, own optimizer, own cleanup.

In [ ]:
run_stage("s1_t4", None, 4, pools[0])


## 11 · Stage S2 — 2 tokens/step, pool 2
Resumes from S1's adapters, trains on **different** 500 problems.

In [ ]:
run_stage("s2_t2", "s1_t4", 2, pools[1])


## 12 · Stage S3 — 1 token/step, pool 3
Resumes from S2. This stage's output carries S1+S2+S3 and is the row to evaluate first.

In [ ]:
run_stage("s3_t1", "s2_t2", 1, pools[2])


## 13 · Benchmark — S3 first, then S1, S2 (raw reused, not re-measured)

Same 40 held-out problems, same scorer, same two schedules as the supervise_all run, so every
number here drops straight into the same table. Two changes, both about deadline economics:

* **`cfg.bench_include_raw = False`** — raw is already measured on this exact harness
  (35% @ t2, 48% @ t1, rep4 0.184/0.185) and cannot have changed. Those numbers are injected
  into the results dict from `cfg` for the summary table, flagged `"reused": true` in the JSON.
  Flip the flag if you want it re-measured anyway.
* **`cfg.bench_order`** puts **S3 first**. If Colab dies mid-benchmark you still have the row
  that decides the paper's framing.

Results → `results/normal_sft_benchmark.json`. Partial results are written after **every**
model, so a crash never costs more than one stage.

In [ ]:
import torch, time, json, re

def score_generation(text, gold):
    def norm_ci(s):
        if s is None: return None
        s = str(s).strip().lower().replace(" ", "")
        for a,b in (("\\left",""),("\\right",""),("\\dfrac","\\frac"),("\\tfrac","\\frac"),("$","")):
            s = s.replace(a,b)
        if s.startswith("\\text{") and s.endswith("}"): s = s[6:-1]
        return s
    def match(a,b):
        a,b = norm_ci(a), norm_ci(b)
        if a is None or b is None: return False
        if a == b: return True
        try: return abs(float(a)-float(b)) < 1e-6
        except Exception: return False
    pred = extract_boxed(text)
    strict_ok = match(pred, gold) if pred else False
    if pred is None:
        tail = text[-300:]
        for pat in (r"(?:the answer is|answer:)\s*\$?([^\.\n$]{1,40})",
                    r"(?:the limit is|equals?)\s*\$?([^\.\n$]{1,40})",
                    r"=\s*\$?([^\.\n$=]{1,25})\s*\$?\.?\s*$"):
            m = re.findall(pat, tail, re.IGNORECASE)
            if m: pred = m[-1].strip(); break
    return {"pred": pred, "correct": match(pred, gold) if pred else False,
            "correct_strict": strict_ok}

@torch.no_grad()
def bench_rollout(model, tok, invalid, mask_id, question, tps):
    dev = model.device
    ids = tok(student_prompt(tok, question), return_tensors="pt").input_ids.to(dev)
    plen = ids.shape[1]; gen, txt, why = 0, "", "budget"
    while gen < cfg.gen_budget:
        cur = ids.shape[1]
        work = torch.cat([ids, torch.full((1,cfg.block_size), mask_id, dtype=ids.dtype, device=dev)], 1)
        pos = torch.arange(cur, cur+cfg.block_size, device=dev)
        still = torch.ones(cfg.block_size, dtype=torch.bool, device=dev)
        while bool(still.any()):
            lg = logits_of(model, work, plen)[0,pos,:].float().masked_fill(invalid, float("-inf"))
            conf = torch.log_softmax(lg,-1).max(-1).values.masked_fill(~still, float("-inf"))
            k = min(tps, int(still.sum().item()))
            sel = conf.topk(k).indices
            work[0,pos[sel]] = lg[sel].argmax(-1); still[sel] = False
        new = work[:,cur:cur+cfg.block_size]
        ids = torch.cat([ids,new],1); gen += cfg.block_size
        g = [t for t in ids[0,plen:].tolist() if t < len(tok)]
        txt = tok.decode(g, skip_special_tokens=True)
        if has_complete_box(txt): why="box"; break
        if tok.eos_token_id is not None and bool((new==tok.eos_token_id).any()): why="eos"; break
        if gen >= 128 and repeat4(txt) > cfg.repeat4_max: why="collapse"; break
    return txt, gen, why

def eval_model(label, model, tok):
    invalid, mask_id = build_invalid(model, tok); model.eval()
    out = {}
    for tps in cfg.bench_schedules:
        rows, t0 = [], time.time()
        for i, p in enumerate(bench):
            txt, n, why = bench_rollout(model, tok, invalid, mask_id, p["question"], tps)
            sc = score_generation(txt, p["gold"])
            rows.append({**sc, "tier": p["tier"], "repeat4": repeat4(txt), "stop": why})
            if (i+1) % 10 == 0:
                nc = sum(r["correct"] for r in rows)
                print(f"    {label} t{tps}: {i+1}/{len(bench)} | correct {nc}/{len(rows)} "
                      f"| {(time.time()-t0)/60:.1f} min")
        d = {"overall": sum(r["correct"] for r in rows)/len(rows),
             "strict": sum(r["correct_strict"] for r in rows)/len(rows),
             "rep4": sum(r["repeat4"] for r in rows)/len(rows)}
        for tier in ("easy","medium","hard"):
            tr = [r for r in rows if r["tier"]==tier]
            d[tier] = sum(r["correct"] for r in tr)/len(tr) if tr else float("nan")
        out[tps] = d
        print(f"  {label} t{tps}: overall {d['overall']:.0%} (strict {d['strict']:.0%}) | "
              f"easy {d['easy']:.0%} med {d['medium']:.0%} hard {d['hard']:.0%} | rep4 {d['rep4']:.3f}")
    return out

results = {}

# ---- raw: reuse the measured numbers unless explicitly re-running ------------
RAW_MEASURED = {2: {"overall":0.35,"strict":0.35,"easy":0.60,"medium":0.10,"hard":0.10,
                    "rep4":0.184,"reused":True},
                1: {"overall":0.48,"strict":0.48,"easy":0.65,"medium":0.40,"hard":0.20,
                    "rep4":0.185,"reused":True}}
if cfg.bench_include_raw:
    print("=== RAW (re-measuring) ===")
    m, tok = load_base(); results["raw"] = eval_model("raw", m, tok)
    del m, tok; import gc; gc.collect(); torch.cuda.empty_cache()
else:
    results["raw"] = {k: dict(v) for k, v in RAW_MEASURED.items()}
    print("=== RAW: reusing measured numbers (35% @t2, 48% @t1) — set "
          "cfg.bench_include_raw=True to re-run ===")
json.dump(results, open(rs("normal_sft_benchmark.json"), "w"), indent=1)

import gc
for stage in cfg.bench_order:
    if not adapters_exist(stage):
        print(f"\n{stage}: no adapters — skipping"); continue
    print(f"\n=== {stage} ===")
    base, tok = load_base()
    peft = PeftModel.from_pretrained(base, ck(stage), is_trainable=False)
    m = peft.merge_and_unload(); m.gradient_checkpointing_disable()
    if hasattr(m.config,"fuse_cross_entropy"): m.config.fuse_cross_entropy = False
    results[stage] = eval_model(stage, m, tok)
    json.dump(results, open(rs("normal_sft_benchmark.json"), "w"), indent=1)   # save per model
    del m, peft, base, tok; gc.collect(); torch.cuda.empty_cache()

json.dump(results, open(rs("normal_sft_benchmark.json"),"w"), indent=1)
print(f"\n{'='*76}\nSUMMARY  (masked-only objective, 3 x 500 disjoint problems)")
ROW_ORDER = ["raw"] + [s for s in ("s1_t4","s2_t2","s3_t1") if s in results]
for tps in cfg.bench_schedules:
    print(f"\n--- tokens_per_step = {tps} ---")
    print(f"{'model':<12}{'overall':>9}{'strict':>9}{'easy':>8}{'medium':>8}{'hard':>8}{'rep4':>8}")
    for name in ROW_ORDER:
        r = results.get(name, {})
        if tps not in r: continue
        d = r[tps]; tag = "  (reused)" if d.get("reused") else ""
        print(f"{name:<12}{d['overall']:>9.0%}{d['strict']:>9.0%}{d['easy']:>8.0%}"
              f"{d['medium']:>8.0%}{d['hard']:>8.0%}{d['rep4']:>8.3f}{tag}")

print(f"\n--- side by side vs supervise_all (t1), for the paper ---")
SA_T1 = {"raw":0.48,"s1_t4":0.20,"s2_t2":0.10,"s3_t1":0.08}
SA_R4 = {"raw":0.185,"s1_t4":0.436,"s2_t2":0.608,"s3_t1":0.497}
print(f"{'model':<10}{'sup_all':>9}{'masked':>9}{'delta':>9}   {'rep4 sup':>9}{'rep4 msk':>9}")
for name in ROW_ORDER:
    if 1 not in results.get(name, {}): continue
    mk = results[name][1]["overall"]; mr = results[name][1]["rep4"]
    sa = SA_T1.get(name); sr = SA_R4.get(name)
    if sa is None: continue
    print(f"{name:<10}{sa:>9.0%}{mk:>9.0%}{mk-sa:>+9.0%}   {sr:>9.3f}{mr:>9.3f}")

print("\nwrote", rs("normal_sft_benchmark.json"))


## 14 · Notes — what to conclude from the table

**Run order.** §1–§9 every session, then §10/§11/§12 (S1→S2→S3), then §13. Each stage skips
itself when `STATE` says done and resumes mid-stage otherwise, so re-running is always safe.

**Judge by §13, not by training loss.** The masked-only loss is *not* comparable to the
supervise_all loss (different supervised sets), and S1/S2/S3 losses are not comparable to each
other either (different schedules ⇒ different sub-step counts).

**The three outcomes and what each one means.**

* **S3 at t1 ≳ 45%** (near or above raw's 48%) — the damage was **supervision density**.
  Dense supervision on already-revealed positions destroys the model; the standard objective
  does not. This is the strongest result available: a controlled single-variable ablation, and
  the paper leads with it.
* **S3 at t1 ≲ 25%** (comparable to supervise_all's 8–20%) — the model is brittle to *any*
  LoRA SFT in this regime. Weaker headline, but it strengthens the compression-floor
  narrative: a 4B block-32 student near the floor cannot absorb further fine-tuning. Required
  caveats: LoRA-only (no full-FT control), one dataset, one model, and the truncation confound
  from §6.
* **In between** — dose–response: supervision density *modulates* damage rather than causing
  it outright. Report the gradient honestly; it is still a real finding.

**Watch these three diagnostics, not just accuracy.**
1. **repeat-4.** In the supervise_all run it rose monotonically (0.185 → 0.436 → 0.608 → 0.497
   at t1). If it stays near 0.19 here, degeneration is objective-driven — the single cleanest
   sentence in the paper.
2. **The `stop` reason** in the per-problem rows (`box` / `eos` / `collapse` / `budget`). The
   supervise_all models stopped terminating, which is why eval wall-clock went 25 min → 74 min.
   A shift back toward `box`/`eos` is mechanistic evidence even if accuracy only partly
   recovers.
3. **Medium tier.** Raw scores 40% @t1 and 10% @t2; supervise_all zeroed it. Medium is where
   both the compression cost and the SFT damage land, so it is the most sensitive column.

**Known confound to disclose (§6).** Solutions longer than `max_target_tokens=384` are
truncated and lose their EOS, so those samples train "do not terminate". This is identical in
both arms, so it does not threaten the comparison — but if *both* arms degrade with the
non-termination signature, truncation is a plausible common cause and must be stated. The cheap
control is a short `require_complete=True` run on ~150 problems at one schedule.